In [1]:
import os
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score
import torchvision.models as models
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode   
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch.nn as nn

In [2]:
DATA_ROOT  = "/kaggle/input/datasets/ipythonx/mvtec-ad"
CATEGORIES = ["bottle", "cable", "tile"]
IMG_SIZE   = 224
SAVE_DIR   = "./patchcore_results"
os.makedirs(SAVE_DIR, exist_ok=True)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {DEVICE}")

Using: cuda


In [3]:
class MVTecSimple(Dataset):

    def __init__(self, root, category, split="train", img_size=224):
        self.samples = []
        base = Path(root) / category
        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
      
        self.mask_transform = T.Compose([
            T.Resize((img_size, img_size), interpolation=InterpolationMode.NEAREST),
            T.ToTensor(),
        ])

        if split == "train":
            for p in sorted((base / "train" / "good").glob("*.png")):
                self.samples.append((str(p), None, 0))
        else:
            test_dir = base / "test"
            mask_dir = base / "ground_truth"
            for defect_type in sorted(test_dir.iterdir()):
                label = 0 if defect_type.name == "good" else 1
                for p in sorted(defect_type.glob("*.png")):
                    mp = mask_dir / defect_type.name / (p.stem + "_mask.png") if label else None
                    self.samples.append((str(p), str(mp) if mp and Path(mp).exists() else None, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ip, mp, lbl = self.samples[idx]
        img  = self.transform(Image.open(ip).convert("RGB"))
        mask = self.mask_transform(Image.open(mp).convert("L")) if mp else torch.zeros(1, IMG_SIZE, IMG_SIZE)
        return img, (mask > 0.5).float(), lbl

In [4]:
class FeatureExtractor(nn.Module):

    def __init__(self):
        super().__init__()
        backbone = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.IMAGENET1K_V1)  # fix #5
        self.layer1 = nn.Sequential(*list(backbone.children())[:5])
        self.layer2 = nn.Sequential(*list(backbone.children())[5:6])
        self.layer3 = nn.Sequential(*list(backbone.children())[6:7])

    def forward(self, x):
        f1 = self.layer1(x)
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        return f2, f3


def extract_patch_features(extractor, dataloader, device):
    extractor.eval()
    all_feats = []
    with torch.no_grad():
        for imgs, _, _ in dataloader:
            f2, f3 = extractor(imgs.to(device))
            f3_up = torch.nn.functional.interpolate(
                f3, size=f2.shape[-2:], mode="bilinear", align_corners=False
            )
            combined = torch.cat([f2, f3_up], dim=1)
            B, C, H, W = combined.shape
            patches = combined.permute(0, 2, 3, 1).reshape(-1, C)
            all_feats.append(patches.cpu())
    return torch.cat(all_feats, dim=0)


def greedy_coreset(features, ratio=0.1):
    n = features.shape[0]
    k = max(1, int(n * ratio))
    selected = [np.random.randint(n)]
    dists = torch.cdist(features[selected[-1]:selected[-1]+1], features).squeeze(0)

    for _ in range(k - 1):
        idx = dists.argmax().item()
        selected.append(idx)
        new_dists = torch.cdist(features[idx:idx+1], features).squeeze(0)
        dists = torch.minimum(dists, new_dists)

    return features[selected]


def run_patchcore_manual(category, data_root=DATA_ROOT, device_str=DEVICE):  
    device = torch.device(device_str)
    print(f"\n[Manual PatchCore]  Category: {category.upper()}")

    train_ds = MVTecSimple(data_root, category, "train", IMG_SIZE)
    test_ds  = MVTecSimple(data_root, category, "test",  IMG_SIZE)
    train_dl = DataLoader(train_ds, batch_size=32, shuffle=False, num_workers=2)
    test_dl  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

    extractor = FeatureExtractor().to(device)

    print("  Extracting training features...")
    train_feats = extract_patch_features(extractor, train_dl, device)
    print(f"  Train feature bank: {train_feats.shape}")

    print("  Running coreset reduction (10%)...")
    memory_bank = greedy_coreset(train_feats, ratio=0.1).to(device)
    print(f"  Memory bank size after reduction: {memory_bank.shape}")

    print("  Scoring test images...")
    extractor.eval()
    image_scores, pixel_maps, labels, gt_masks = [], [], [], []

    with torch.no_grad():
        for imgs, masks, lbls in test_dl:
            imgs = imgs.to(device)
            f2, f3 = extractor(imgs)
            f3_up = torch.nn.functional.interpolate(
                f3, size=f2.shape[-2:], mode="bilinear", align_corners=False
            )
            combined = torch.cat([f2, f3_up], dim=1)
            B, C, H, W = combined.shape
            patches = combined.permute(0, 2, 3, 1).reshape(B, H * W, C)

            scores_map = []
            for b in range(B):
                dists  = torch.cdist(patches[b], memory_bank)
                nn_dist = dists.min(dim=1).values
                scores_map.append(nn_dist.reshape(H, W))
            scores_map = torch.stack(scores_map)

            img_score = scores_map.reshape(B, -1).max(dim=1).values

            score_map_full = torch.nn.functional.interpolate(
                scores_map.unsqueeze(1), size=(IMG_SIZE, IMG_SIZE),
                mode="bilinear", align_corners=False
            ).squeeze(1)

            image_scores.append(img_score.cpu().numpy())
            pixel_maps.append(score_map_full.cpu().numpy())
            labels.append(lbls.numpy())
            gt_masks.append(masks.squeeze(1).numpy())

    image_scores = np.concatenate(image_scores)
    pixel_maps   = np.concatenate(pixel_maps)
    labels       = np.concatenate(labels)
    gt_masks     = np.concatenate(gt_masks)

    img_auroc = roc_auc_score(labels, image_scores)
    has_mask  = gt_masks.sum(axis=(1, 2)) > 0
    pix_auroc = roc_auc_score(gt_masks[has_mask].flatten(), pixel_maps[has_mask].flatten())

    print(f"\n  Image AUROC: {img_auroc*100:.2f}%")
    print(f"  Pixel AUROC: {pix_auroc*100:.2f}%")

    return {"category": category, "img_auroc": img_auroc, "pix_auroc": pix_auroc}




In [5]:
if __name__ == "__main__":
    all_results = {}

    for cat in CATEGORIES:
        result = None  
        try:
            result = run_patchcore_manual(cat)
        except Exception as e:
            print(f"  Error on category '{cat}': {e}")

      
        if result is not None:
            all_results[cat] = result

    print("\n       PatchCore FINAL RESULTS SUMMARY        ")
    print(" Category              Img AUROC  Pix AUROC ")
    for cat, m in all_results.items():
        print(f"║ {cat:<20} ║   {m['img_auroc']*100:.2f}%  ║   {m['pix_auroc']*100:.2f}%  ║")

    with open("patchcore_all_results.json", "w") as f:
        json.dump(all_results, f, indent=2)


[Manual PatchCore]  Category: BOTTLE
Downloading: "https://download.pytorch.org/models/wide_resnet50_2-95faca4d.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet50_2-95faca4d.pth
  Error on category 'bottle': <urlopen error [Errno -3] Temporary failure in name resolution>

[Manual PatchCore]  Category: CABLE
Downloading: "https://download.pytorch.org/models/wide_resnet50_2-95faca4d.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet50_2-95faca4d.pth
  Error on category 'cable': <urlopen error [Errno -3] Temporary failure in name resolution>

[Manual PatchCore]  Category: TILE
Downloading: "https://download.pytorch.org/models/wide_resnet50_2-95faca4d.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet50_2-95faca4d.pth
  Error on category 'tile': <urlopen error [Errno -3] Temporary failure in name resolution>

       PatchCore FINAL RESULTS SUMMARY        
 Category              Img AUROC  Pix AUROC 
